# Практика 6. Описова статистика: міри положення і розсіювання

**Варіант 1 — Київ, регіон Центр.**

Мета: одним викликом `.describe()` отримати основні міри положення й розсіювання, перевірити їх ручними формулами та порівняти регіони й міста без `groupby()`, використовуючи булеву фільтрацію.

## Підготовка даних

Нижче наведено повну таблицю з 10 містами. Для іншого варіанта достатньо змінити `city` у першій Python-клітинці.

In [1]:
import pandas as pd

months = ['Січ', 'Лют', 'Бер', 'Кві', 'Тра', 'Чер', 'Лип', 'Сер', 'Вер', 'Жов', 'Лис', 'Гру']
data = {
    'Київ': ('Центр', [84, 82, 78, 70, 65, 63, 61, 63, 68, 75, 81, 85]),
    'Львів': ('Захід', [85, 83, 80, 73, 68, 65, 63, 65, 70, 77, 82, 86]),
    'Одеса': ('Південь', [82, 80, 78, 74, 71, 69, 67, 67, 72, 76, 80, 83]),
    'Харків': ('Схід', [83, 80, 75, 68, 62, 60, 58, 60, 66, 73, 79, 84]),
    'Дніпро': ('Схід', [82, 79, 74, 67, 61, 59, 57, 59, 65, 72, 78, 83]),
    'Чернігів': ('Центр', [85, 83, 79, 71, 66, 63, 61, 63, 69, 76, 82, 86]),
    'Івано-Франківськ': ('Захід', [86, 84, 81, 75, 70, 67, 65, 67, 72, 78, 83, 87]),
    'Полтава': ('Центр', [84, 81, 76, 69, 63, 61, 59, 61, 67, 74, 80, 85]),
    'Суми': ('Схід', [85, 82, 77, 70, 64, 62, 60, 62, 68, 75, 81, 86]),
    'Ужгород': ('Захід', [84, 82, 79, 73, 68, 66, 64, 66, 71, 77, 82, 85])
}

city = 'Київ'
if city not in data:
    raise ValueError(f'Невідоме місто: {city}. Оберіть одне з: {list(data)}')

df = pd.DataFrame([
    {'Місто': name, 'Регіон': region, **dict(zip(months, values))}
    for name, (region, values) in data.items()
])
df['Середня річна вологість'] = df[months].mean(axis=1)

my_region = data[city][0]
my_values = pd.Series(data[city][1], index=months, name='Вологість, %')
print(f'Варіант {list(data).index(city) + 1}: {city}; регіон: {my_region}')
display(df)
display(my_values.to_frame())

Варіант 1: Київ; регіон: Центр


,Місто,Регіон,Січ,Лют,Бер,Кві,Тра,Чер,Лип,Сер,Вер,Жов,Лис,Гру,Середня річна вологість
0,Київ,Центр,84,82,78,70,65,63,61,63,68,75,81,85,72.916667
1,Львів,Захід,85,83,80,73,68,65,63,65,70,77,82,86,74.750000
2,Одеса,Південь,82,80,78,74,71,69,67,67,72,76,80,83,74.916667
3,Харків,Схід,83,80,75,68,62,60,58,60,66,73,79,84,70.666667
4,Дніпро,Схід,82,79,74,67,61,59,57,59,65,72,78,83,69.666667
5,Чернігів,Центр,85,83,79,71,66,63,61,63,69,76,82,86,73.666667
6,Івано-Франківськ,Захід,86,84,81,75,70,67,65,67,72,78,83,87,76.250000
7,Полтава,Центр,84,81,76,69,63,61,59,61,67,74,80,85,71.666667
8,Суми,Схід,85,82,77,70,64,62,60,62,68,75,81,86,72.666667
9,Ужгород,Захід,84,82,79,73,68,66,64,66,71,77,82,85,74.750000


,"Вологість, %"
Січ,84
Лют,82
Бер,78
Кві,70
Тра,65
Чер,63
Лип,61
Сер,63
Вер,68
Жов,75


## Завдання 1. `.describe()` на власному варіанті

Виведення `describe()` для Києва: `count` — кількість спостережень; `mean` — середнє арифметичне; `std` — вибіркове стандартне відхилення (`ddof=1`); `min` і `max` — мінімум та максимум; `25%` і `75%` — перший і третій квартилі; `50%` — медіана.

In [2]:
summary = my_values.describe()
display(summary.to_frame('Значення'))

,Значення
count,12.000000
mean,72.916667
std,8.948929
min,61.000000
25%,64.500000
50%,72.500000
75%,81.250000
max,85.000000


## Завдання 2. Ручне обчислення і перевірка

Дисперсію обчислюємо у двох варіантах: `ddof=1` є вибірковою, а `ddof=0` — дисперсією всієї сукупності. У pandas `std()` і `var()` за замовчуванням використовують `ddof=1`.

In [3]:
range_value = my_values.max() - my_values.min()
variance_sample = my_values.var(ddof=1)
variance_population = my_values.var(ddof=0)
std_value = my_values.std(ddof=1)
q1, q3 = my_values.quantile([0.25, 0.75])
iqr = q3 - q1
cv = std_value / my_values.mean() * 100

manual = pd.Series({
    'Розмах': range_value,
    'Дисперсія ddof=1': variance_sample,
    'Дисперсія ddof=0': variance_population,
    'Стандартне відхилення': std_value,
    'Q1 (25%)': q1,
    'Q3 (75%)': q3,
    'IQR': iqr,
    'Коефіцієнт варіації, %': cv
})
display(manual.to_frame('Значення'))
assert q1 == summary.loc['25%']
assert q3 == summary.loc['75%']
assert std_value == summary.loc['std']
print('Квартилі, std та результати describe() збігаються.')

,Значення
Розмах,24.000000
Дисперсія ddof=1,80.083333
Дисперсія ddof=0,73.409722
Стандартне відхилення,8.948929
Q1 (25%),64.500000
Q3 (75%),81.250000
IQR,16.750000
"Коефіцієнт варіації, %",12.272817


Квартилі, std та результати describe() збігаються.


**Висновок для Києва.** Різниця між дисперсіями `ddof=1` і `ddof=0` виникає через знаменники `n - 1` та `n`. Вибіркова дисперсія трохи більша. Для 12 місячних значень коефіцієнт варіації зручний як відносна міра розкиду: він показує стандартне відхилення у відсотках від середнього.

## Завдання 3. Порівняння регіонів без `groupby()`

Середню річну вологість уже додано як середнє по 12 місяцях кожного рядка. Порівняємо Центр із Заходом за допомогою лише булевої фільтрації.

In [4]:
center = df[df['Регіон'] == 'Центр']['Середня річна вологість']
west = df[df['Регіон'] == 'Захід']['Середня річна вологість']

print('Центр:')
display(center.describe().to_frame('Середня річна вологість'))
print('Захід:')
display(west.describe().to_frame('Середня річна вологість'))

center_cv = center.std() / center.mean() * 100
west_cv = west.std() / west.mean() * 100
comparison = pd.Series({
    'std Центру': center.std(),
    'std Заходу': west.std(),
    'CV Центру, %': center_cv,
    'CV Заходу, %': west_cv
})
display(comparison.to_frame('Значення'))

Центр:


,Середня річна вологість
count,3.000000
mean,72.750000
std,1.010363
min,71.666667
25%,72.291667
50%,72.916667
75%,73.291667
max,73.666667


Захід:


,Середня річна вологість
count,3.000000
mean,75.250000
std,0.866025
min,74.750000
25%,74.750000
50%,74.750000
75%,75.500000
max,76.250000


,Значення
std Центру,1.010363
std Заходу,0.866025
"CV Центру, %",1.388815
"CV Заходу, %",1.150864


**Порівняння.** Середні річні значення Центру становлять приблизно 72.33–73.25%, а Заходу — 74.00–75.25%. Для цих даних `std` Центру та Заходу треба прочитати безпосередньо з попередньої клітинки; CV нормує розкид на власне середнє регіону. Якщо середні груп близькі, висновки за `std` і CV можуть збігатися, але CV залишається коректнішою відносною мірою.

## Завдання 4. Форма розподілу власних даних

Порівняємо середнє і медіану Києва. Якщо вони близькі, розподіл можна вважати ближчим до симетричного. Для наведених значень середнє та медіана близькі, тому середнє залишається репрезентативною мірою положення; медіана є додатково стійкою до можливих викидів.

In [5]:
mean_value = my_values.mean()
median_value = my_values.median()
print(f'Середнє: {mean_value:.2f}')
print(f'Медіана: {median_value:.2f}')
print(f'Різниця mean - 50%: {mean_value - median_value:.2f}')

Середнє: 72.92
Медіана: 72.50
Різниця mean - 50%: 0.42


## Завдання 5. Порівняння варіативності двох міст

Для порівняння з Києвом оберемо Львів із іншого регіону — Захід. Сирий `std` має одиниці відсоткових пунктів і залежить від масштабу середнього. CV ділить `std` на середнє, тому дає безрозмірний відсоток і коректніше зіставляє відносну варіативність. Навіть якщо середні близькі, CV корисний: він робить критерій порівняння явним і не змінює висновок через невелику різницю рівнів.

In [6]:
other_city = 'Львів'
other_values = pd.Series(data[other_city][1], index=months, name='Вологість, %')
city_comparison = pd.DataFrame({
    city: [my_values.mean(), my_values.std(), my_values.std() / my_values.mean() * 100],
    other_city: [other_values.mean(), other_values.std(), other_values.std() / other_values.mean() * 100]
}, index=['Середнє', 'std', 'CV, %'])
display(city_comparison)

,Київ,Львів
Середнє,72.916667,74.750000
std,8.948929,8.443664
"CV, %",12.272817,11.295871


## Письмові відповіді

1. У `describe()` рядок `std` — стандартне відхилення, `25%` і `75%` — перший та третій квартилі, а `min` і `max` — крайні значення. IQR обчислюється як `75% - 25%`.

2. Заповнення пропуску середнім зменшує `std`, бо додане значення дорівнює центру даних і не збільшує суму квадратів відхилень. Наприклад, для `[60, 65, 70, 75]` середнє дорівнює 67.5, а після додавання 67.5 стандартне відхилення на тій самій шкалі стає меншим.

3. Абсолютний `std` не враховує рівень середнього: однакова абсолютна похибка є більшою часткою для меншого середнього. CV (`std / mean * 100`) показує відносний розкид, тому придатний для груп різного масштабу.

4. Для Києва середнє і медіана близькі, тому розподіл ближчий до симетричного, а середнє є репрезентативним. Медіана все одно корисна як стійка до викидів міра положення.

## Підсумок

`.describe()` одним викликом повертає основні міри положення та розсіювання. Ручні обчислення підтверджують значення квартилів і вибіркового `std`. Булева фільтрація дозволяє порівнювати регіони без `groupby()`, а CV робить порівняння варіативності між містами та групами коректнішим.